ImageFolder, transforms и DataLoader

In [ ]:
train_dir = DATA_DIR / "train"
val_dir = DATA_DIR / "val"
test_dir = DATA_DIR / "test"
image_size = 224
batch_size = 16
basic_transform = transforms.Compose([transforms.Resize((image_size, image_size)),transforms.ToTensor()])
train_dataset = datasets.ImageFolder(train_dir, transform=basic_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=basic_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=basic_transform)
loader_kwargs = {
    "num_workers": 0,
    "pin_memory": torch.cuda.is_available()
}

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, **loader_kwargs)
num_classes = len(train_dataset.classes)
print("Классы:", train_dataset.classes)
print("количество классов", num_classes)

Классы: ['crack', 'dent', 'glass_shatter', 'lamp_broken', 'scratch', 'tire_flat']
количество классов 6


In [ ]:
class_mapping_path = PROCESSED_DIR / "class_to_idx.json"
with open(class_mapping_path, "w") as f:
    json.dump(train_dataset.class_to_idx, f)

ЛОГИРОВАНИЕ!!!!!

In [ ]:
if USE_WANDB:
    !pip install -q wandb
    import wandb
    wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: WARNING Invalid choice


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: af870295 (af870295-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
def log_dataset_artifact():
    if not USE_WANDB or not LOG_DATASET_ARTIFACT:
        return
    wandb.init(project=PROJECT_NAME,name="cardd_crops_dataset_artifact",job_type="dataset_logging")
    artifact = wandb.Artifact(name=DATASET_ARTIFACT_NAME,type="dataset",description="Processed CarDD crop dataset")
    artifact.add_file(str(ARCHIVE_PATH))
    artifact.add_file(str(metadata_path))
    artifact.add_file(str(class_mapping_path))
    wandb.log_artifact(artifact)
    wandb.finish()

def use_dataset_artifact():
    if USE_WANDB:
        wandb.use_artifact(f"{DATASET_ARTIFACT_NAME}:latest", type="dataset")
log_dataset_artifact()

wandb: WARNING Artifact "cardd_crops_train_val_test" already exists with the same content. No new version will be created.
